<a href="https://colab.research.google.com/github/raihanpakbar/CV/blob/master/Algoritma_RSA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import sys

def jeda(detik=1.5):
    time.sleep(detik)

def cetak_pelan(teks, detik=0.05):
    for huruf in teks:
        sys.stdout.write(huruf)
        sys.stdout.flush()
        time.sleep(detik)
    print()

def demo_rsa_visual():
    p, q = 17, 19
    n = p * q
    e = 5
    d = 173

    print("\n[1] KEY GENERATION PROCESS")
    cetak_pelan(f"-> Membangun kunci dari bilangan prima p={p}, q={q}")
    jeda()
    cetak_pelan(f"-> Public Key  (e, n) : ({e}, {n})  <-- Dibagikan ke publik")
    jeda()
    cetak_pelan(f"-> Private Key (d, n) : ({d}, {n}) <-- Disimpan rahasia")
    jeda(2)

    # 2. INPUT USER
    print("\n[2] MESSAGE INPUT")
    pesan = input("Masukkan teks rahasia (contoh: HALO): ")
    jeda(1)

    # 3. ENCRYPTION PROCESS
    print("\n[3] ENCRYPTION PROCESS (Menggunakan Public Key)")
    cetak_pelan("Formula: Cipher = (ASCII_Char ^ e) mod n")
    jeda(1)

    cipherteks_list = []
    for char in pesan:
        m = ord(char)
        c = (m ** e) % n
        cipherteks_list.append(c)
        char = chr(c)

        print(f"Mengenkripsi '{char}' (ASCII: {m}) ...")
        time.sleep(0.8)
        print(f"  -> {m}^{e} mod {n} = {c}: {char}") # Hasilnya sekarang pasti berbeda!
        time.sleep(1.2)

    print(f"\n=> Hasil Ciphertext (Angka acak): {cipherteks_list}")
    jeda(2)

    # 4. DECRYPTION PROCESS
    print("\n[4] DECRYPTION PROCESS (Menggunakan Private Key)")
    cetak_pelan("Formula: Plaintext = (Cipher ^ d) mod n")
    jeda(1)

    teks_terdekripsi = ""
    for c in cipherteks_list:
        print(f"Mendekripsi cipher '{c}' ...")
        time.sleep(0.8)

        m = (c ** d) % n
        char = chr(m)
        teks_terdekripsi += char

        print(f"  -> {c}^{d} mod {n} = {m} (Karakter: '{char}')")
        time.sleep(1.2)

    print("\n[5] FINAL RESULT")
    cetak_pelan(f"Pesan Asli       : {pesan}")
    cetak_pelan(f"Pesan Terdekripsi: {teks_terdekripsi}")

    if pesan == teks_terdekripsi:
        print("\nALGORITMA BERJALAN SEMPURNA")

if __name__ == "__main__":
    demo_rsa_visual()


[1] KEY GENERATION PROCESS
-> Membangun kunci dari bilangan prima p=17, q=19
-> Public Key  (e, n) : (5, 323)  <-- Dibagikan ke publik
-> Private Key (d, n) : (173, 323) <-- Disimpan rahasia

[2] MESSAGE INPUT
Masukkan teks rahasia (contoh: HALO): arigatou

[3] ENCRYPTION PROCESS (Menggunakan Public Key)
Formula: Cipher = (ASCII_Char ^ e) mod n
Mengenkripsi 'ñ' (ASCII: 97) ...
  -> 97^5 mod 323 = 241: ñ
Mengenkripsi '¾' (ASCII: 114) ...
  -> 114^5 mod 323 = 190: ¾
Mengenkripsi '' (ASCII: 105) ...
  -> 105^5 mod 323 = 22: 
Mengenkripsi 'E' (ASCII: 103) ...
  -> 103^5 mod 323 = 69: E
Mengenkripsi 'ñ' (ASCII: 97) ...
  -> 97^5 mod 323 = 241: ñ
Mengenkripsi '¥' (ASCII: 116) ...
  -> 116^5 mod 323 = 165: ¥
Mengenkripsi '*' (ASCII: 111) ...
  -> 111^5 mod 323 = 42: *
Mengenkripsi '5' (ASCII: 117) ...
  -> 117^5 mod 323 = 53: 5

=> Hasil Ciphertext (Angka acak): [241, 190, 22, 69, 241, 165, 42, 53]

[4] DECRYPTION PROCESS (Menggunakan Private Key)
Formula: Plaintext = (Cipher ^ d) mod n
Me

In [ ]:
import os
import base64
import binascii
from cryptography.hazmat.primitives.asymmetric import rsa

def potong(teks, batas=15):
    teks = str(teks)
    if len(teks) > batas * 2:
        return f"{teks[:batas]} ... {teks[-batas:]}"
    return teks

def main():
    print("--- SETUP KUNCI ---")
    print("Generasi RSA 1024-bit (128 bytes)...")
    priv = rsa.generate_private_key(public_exponent=65537, key_size=1024)
    pub = priv.public_key()

    n = pub.public_numbers().n
    e = pub.public_numbers().e
    d = priv.private_numbers().d
    k = 128
    print("Selesai.\n")

    teks_asli = input("Input Plaintext: ")
    pesan_bytes = teks_asli.encode('utf-8')
    raw_hex = binascii.hexlify(pesan_bytes).decode('utf-8')

    # ==========================================
    # PROSES ENKRIPSI
    # ==========================================
    print("\n=== PIPELINE ENKRIPSI ===")

    # 1. Padding
    pad_len = k - len(pesan_bytes) - 3
    pad_acak = bytes([b if b != 0 else 1 for b in os.urandom(pad_len)])
    pad_hex = binascii.hexlify(pad_acak).decode('utf-8')

    # Membangun blok data yang utuh
    blok_padded = b'\x00\x02' + pad_acak + b'\x00' + pesan_bytes

    # Visualisasi struktur injeksi
    struktur_injeksi = f"00 02 [{pad_hex[:8]}...{pad_hex[-8:]}] 00 {raw_hex}"

    # 2. OS2IP (Bytes ke Integer)
    m_int = int.from_bytes(blok_padded, 'big')

    # 3. RSA Math
    c_int = pow(m_int, e, n)

    # 4. I2OSP (Integer ke Bytes)
    cipher_bytes = c_int.to_bytes(k, 'big')
    cipher_b64 = base64.b64encode(cipher_bytes).decode('utf-8')

    print(f"1. Teks Asli   : {teks_asli}")
    print(f"2. Raw Bytes   : {raw_hex} (Format Hexadesimal)")
    print(f"3. Injeksi Pad : {struktur_injeksi}")
    print("                 ^^ ^^ ^^^^^^^^^^^^^^^^^^^^^ ^^\n                 |  |  [ Byte Acak/Padding ] |  |-- Raw Bytes\n                 Header                      Pemisah")
    print(f"4. Integer (M) : {potong(m_int, 15)}")
    print(f"5. Integer (C) : {potong(c_int, 15)} (Hasil dari M^e mod n)")
    print(f"6. Cipher B64  : {potong(cipher_b64, 20)}")

    input("\n[Tekan Enter untuk memulai Dekripsi...]")

    # ==========================================
    # PROSES DEKRIPSI
    # ==========================================
    print("\n=== PIPELINE DEKRIPSI ===")

    dec_cipher_bytes = base64.b64decode(cipher_b64)
    dec_c_int = int.from_bytes(dec_cipher_bytes, 'big')
    dec_m_int = pow(dec_c_int, d, n)

    dec_blok_padded = dec_m_int.to_bytes(k, 'big')

    # Ekstraksi komponen untuk visualisasi dekripsi
    dec_pad_hex = binascii.hexlify(dec_blok_padded[2:-len(pesan_bytes)-1]).decode('utf-8')
    dec_raw_hex = binascii.hexlify(dec_blok_padded[-len(pesan_bytes):]).decode('utf-8')
    dec_struktur_injeksi = f"00 02 [{dec_pad_hex[:8]}...{dec_pad_hex[-8:]}] 00 {dec_raw_hex}"

    batas_pemisah = dec_blok_padded.find(b'\x00', 2)
    dec_pesan_bytes = dec_blok_padded[batas_pemisah + 1:]
    hasil_teks = dec_pesan_bytes.decode('utf-8')

    print(f"1. Cipher B64  : {potong(cipher_b64, 20)}")
    print(f"2. Integer (C) : {potong(dec_c_int, 15)}")
    print(f"3. Integer (M) : {potong(dec_m_int, 15)}")
    print(f"4. Ekstraksi   : {dec_struktur_injeksi}")
    print(f"5. Raw Bytes   : {dec_raw_hex} (Padding & Header dibuang)")
    print(f"6. Teks Asli   : {hasil_teks}")

if __name__ == "__main__":
    main()

--- SETUP KUNCI ---
Generasi RSA 1024-bit (128 bytes)...
Selesai.

Input Plaintext: ali ganteng

=== PIPELINE ENKRIPSI ===
1. Teks Asli   : ali ganteng
2. Raw Bytes   : 616c692067616e74656e67 (Format Hexadesimal)
3. Injeksi Pad : 00 02 [4be26267...d75b4cf4] 00 616c692067616e74656e67
                 ^^ ^^ ^^^^^^^^^^^^^^^^^^^^^ ^^
                 |  |  [ Byte Acak/Padding ] |  |-- Raw Bytes
                 Header                      Pemisah
4. Integer (M) : 629923102531376 ... 032728769785447
5. Integer (C) : 340705537738373 ... 175966895241275 (Hasil dari M^e mod n)
6. Cipher B64  : MISg1HbRBU43VQ6oqoPQ ... I46bPHjZlF2v0R50eDs=

[Tekan Enter untuk memulai Dekripsi...]

=== PIPELINE DEKRIPSI ===
1. Cipher B64  : MISg1HbRBU43VQ6oqoPQ ... I46bPHjZlF2v0R50eDs=
2. Integer (C) : 340705537738373 ... 175966895241275
3. Integer (M) : 629923102531376 ... 032728769785447
4. Ekstraksi   : 00 02 [4be26267...d75b4cf4] 00 616c692067616e74656e67
5. Raw Bytes   : 616c692067616e74656e67 (Padding & He

In [ ]:
import os
import base64
import binascii
from cryptography.hazmat.primitives.asymmetric import rsa

def potong(teks, batas=15):
    teks = str(teks)
    if len(teks) > batas * 2:
        return f"{teks[:batas]} ... {teks[-batas:]}"
    return teks

def main():
    print("--- SETUP KUNCI ---")
    print("Generasi RSA 1024-bit (128 bytes)...")
    priv = rsa.generate_private_key(public_exponent=65537, key_size=1024)
    pub = priv.public_key()

    n = pub.public_numbers().n
    e = pub.public_numbers().e
    d = priv.private_numbers().d
    k = 128
    print("Selesai.\n")

    teks_asli = input("Input Plaintext: ")
    pesan_bytes = teks_asli.encode('utf-8')
    raw_hex = binascii.hexlify(pesan_bytes).decode('utf-8')

    # ==========================================
    # PROSES ENKRIPSI
    # ==========================================
    print("\n=== PIPELINE ENKRIPSI ===")

    # 1. Padding
    pad_len = k - len(pesan_bytes) - 3
    pad_acak = bytes([b if b != 0 else 1 for b in os.urandom(pad_len)])
    pad_hex = binascii.hexlify(pad_acak).decode('utf-8')

    # Membangun blok data yang utuh
    blok_padded = b'\x00\x02' + pad_acak + b'\x00' + pesan_bytes

    # Visualisasi struktur injeksi
    struktur_injeksi = f"00 02 [{pad_hex[:8]}...{pad_hex[-8:]}] 00 {raw_hex}"

    # 2. OS2IP (Bytes ke Integer)
    m_int = int.from_bytes(blok_padded, 'big')

    # 3. RSA Math
    c_int = pow(m_int, e, n)

    # 4. I2OSP (Integer ke Bytes)
    cipher_bytes = c_int.to_bytes(k, 'big')
    cipher_b64 = base64.b64encode(cipher_bytes).decode('utf-8')

    print(f"1. Teks Asli   : {teks_asli}")
    print(f"2. Raw Bytes   : {raw_hex} (Format Hexadesimal)")
    print(f"3. Injeksi Pad : {struktur_injeksi}")
    print("                 ^^ ^^ ^^^^^^^^^^^^^^^^^^^^^ ^^\n                 |  |  [ Byte Acak/Padding ] |  |-- Raw Bytes\n                 Header                      Pemisah")
    print(f"4. Integer (M) : {potong(m_int, 15)}")
    print(f"5. Integer (C) : {potong(c_int, 15)} (Hasil dari M^e mod n)")
    print(f"6. Cipher B64  : {potong(cipher_b64, 20)}")

    input("\n[Tekan Enter untuk memulai Dekripsi...]")

    # ==========================================
    # PROSES DEKRIPSI
    # ==========================================
    print("\n=== PIPELINE DEKRIPSI ===")

    dec_cipher_bytes = base64.b64decode(cipher_b64)
    dec_c_int = int.from_bytes(dec_cipher_bytes, 'big')
    dec_m_int = pow(dec_c_int, d, n)

    dec_blok_padded = dec_m_int.to_bytes(k, 'big')

    # Ekstraksi komponen untuk visualisasi dekripsi
    dec_pad_hex = binascii.hexlify(dec_blok_padded[2:-len(pesan_bytes)-1]).decode('utf-8')
    dec_raw_hex = binascii.hexlify(dec_blok_padded[-len(pesan_bytes):]).decode('utf-8')
    dec_struktur_injeksi = f"00 02 [{dec_pad_hex[:8]}...{dec_pad_hex[-8:]}] 00 {dec_raw_hex}"

    batas_pemisah = dec_blok_padded.find(b'\x00', 2)
    dec_pesan_bytes = dec_blok_padded[batas_pemisah + 1:]
    hasil_teks = dec_pesan_bytes.decode('utf-8')

    print(f"1. Cipher B64  : {potong(cipher_b64, 20)}")
    print(f"2. Integer (C) : {potong(dec_c_int, 15)}")
    print(f"3. Integer (M) : {potong(dec_m_int, 15)}")
    print(f"4. Ekstraksi   : {dec_struktur_injeksi}")
    print(f"5. Raw Bytes   : {dec_raw_hex} (Padding & Header dibuang)")
    print(f"6. Teks Asli   : {hasil_teks}")

if __name__ == "__main__":
    main()

--- SETUP KUNCI ---
Generasi RSA 1024-bit (128 bytes)...
Selesai.

Input Plaintext: ali ganteng

=== PIPELINE ENKRIPSI ===
1. Teks Asli   : ali ganteng
2. Raw Bytes   : 616c692067616e74656e67 (Format Hexadesimal)
3. Injeksi Pad : 00 02 [49530114...9b9571a0] 00 616c692067616e74656e67
                 ^^ ^^ ^^^^^^^^^^^^^^^^^^^^^ ^^
                 |  |  [ Byte Acak/Padding ] |  |-- Raw Bytes
                 Header                      Pemisah
4. Integer (M) : 627179956019370 ... 327567233412711
5. Integer (C) : 747287904181544 ... 565301119756456 (Hasil dari M^e mod n)
6. Cipher B64  : amrWIpyRpttRKGZUHJ6C ... OChP+ecA+U9AIOshKKg=

[Tekan Enter untuk memulai Dekripsi...]

=== PIPELINE DEKRIPSI ===
1. Cipher B64  : amrWIpyRpttRKGZUHJ6C ... OChP+ecA+U9AIOshKKg=
2. Integer (C) : 747287904181544 ... 565301119756456
3. Integer (M) : 627179956019370 ... 327567233412711
4. Ekstraksi   : 00 02 [49530114...9b9571a0] 00 616c692067616e74656e67
5. Raw Bytes   : 616c692067616e74656e67 (Padding & He